In [ ]:
"""
ResNet from Scratch on CIFAR-100 — targeting >80% top-1 accuracy
================================================================
Architecture : ResNet-34 (CIFAR-adapted: no max-pool, smaller stem)
Augmentation : RandAugment + Cutout + MixUp + CutMix
Regularization: Label smoothing, Dropout, Weight decay
Schedule     : Cosine annealing with linear warmup
Optimizer    : SGD with Nesterov momentum (or AdamW if preferred)

Expected results
  ~200 epochs → 77–79%
  ~300 epochs → 80–82%

Usage
  python train_cifar100.py                         # defaults
  python train_cifar100.py --epochs 300 --batch 128
  python train_cifar100.py --arch resnet50_cifar   # wider variant
"""

import argparse
import math
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import AutoAugment, AutoAugmentPolicy

# ─────────────────────────────────────────────────────────────────────────────
# 1.  Architecture
# ─────────────────────────────────────────────────────────────────────────────

class BasicBlock(nn.Module):
    """2-layer residual block with optional shortcut projection."""
    expansion = 1

    def __init__(self, in_ch, out_ch, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.drop  = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()

        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.drop(out)
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)


class Bottleneck(nn.Module):
    """3-layer bottleneck block for wider/deeper variants."""
    expansion = 4

    def __init__(self, in_ch, out_ch, stride=1, dropout=0.0):
        super().__init__()
        mid = out_ch
        self.conv1 = nn.Conv2d(in_ch,       mid,            1, bias=False)
        self.bn1   = nn.BatchNorm2d(mid)
        self.conv2 = nn.Conv2d(mid,          mid,            3, stride=stride, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(mid)
        self.conv3 = nn.Conv2d(mid,          out_ch * 4,    1, bias=False)
        self.bn3   = nn.BatchNorm2d(out_ch * 4)
        self.drop  = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()

        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch * 4:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch * 4, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch * 4),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.drop(out)
        out = self.bn3(self.conv3(out))
        out += self.shortcut(x)
        return F.relu(out)


class ResNetCIFAR(nn.Module):
    """
    CIFAR-adapted ResNet.

    Key difference from ImageNet ResNet:
      • 3×3 stem conv (not 7×7), stride=1 (not 2)
      • No max-pool after stem
    This preserves spatial resolution for small inputs.
    """

    CONFIGS = {
        # name          : (block,       [layers],      [channels])
        "resnet18_cifar": (BasicBlock,  [2, 2, 2, 2],  [64, 128, 256, 512]),
        "resnet34_cifar": (BasicBlock,  [3, 4, 6, 3],  [64, 128, 256, 512]),
        "resnet50_cifar": (Bottleneck,  [3, 4, 6, 3],  [64, 128, 256, 512]),
        # Wide variant — more params, higher ceiling
        "wrn28_cifar"   : (BasicBlock,  [4, 4, 4, 4],  [128, 256, 512, 512]),
    }

    def __init__(self, arch="resnet34_cifar", num_classes=100, dropout=0.0, head_dropout=0.3):
        super().__init__()
        block, layers, channels = self.CONFIGS[arch]

        self.stem = nn.Sequential(
            nn.Conv2d(3, channels[0], 3, padding=1, bias=False),
            nn.BatchNorm2d(channels[0]),
            nn.ReLU(inplace=True),
        )

        self.layer1 = self._make_layer(block, channels[0], channels[0], layers[0], stride=1, dropout=dropout)
        self.layer2 = self._make_layer(block, channels[0] * block.expansion, channels[1], layers[1], stride=2, dropout=dropout)
        self.layer3 = self._make_layer(block, channels[1] * block.expansion, channels[2], layers[2], stride=2, dropout=dropout)
        self.layer4 = self._make_layer(block, channels[2] * block.expansion, channels[3], layers[3], stride=2, dropout=dropout)

        final_ch = channels[3] * block.expansion
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(head_dropout)
        self.fc   = nn.Linear(final_ch, num_classes)

        self._init_weights()

    @staticmethod
    def _make_layer(block, in_ch, out_ch, n_blocks, stride, dropout):
        layers = [block(in_ch, out_ch, stride=stride, dropout=dropout)]
        for _ in range(1, n_blocks):
            layers.append(block(out_ch * block.expansion, out_ch, dropout=dropout))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x).flatten(1)
        x = self.drop(x)
        return self.fc(x)


# ─────────────────────────────────────────────────────────────────────────────
# 2.  Augmentation helpers (MixUp & CutMix)
# ─────────────────────────────────────────────────────────────────────────────

def mixup_data(x, y, alpha=0.4):
    """Returns mixed inputs and pairs of targets + lambda."""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam


def cutmix_data(x, y, alpha=1.0):
    """CutMix augmentation — cuts a patch from one image into another."""
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    B, C, H, W = x.shape

    cut_rat = math.sqrt(1.0 - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)
    cx = random.randint(0, W)
    cy = random.randint(0, H)
    x1 = max(cx - cut_w // 2, 0)
    y1 = max(cy - cut_h // 2, 0)
    x2 = min(cx + cut_w // 2, W)
    y2 = min(cy + cut_h // 2, H)

    mixed_x = x.clone()
    mixed_x[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    return mixed_x, y, y[idx], lam


def mixed_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


# ─────────────────────────────────────────────────────────────────────────────
# 3.  Data loaders
# ─────────────────────────────────────────────────────────────────────────────

CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD  = (0.2675, 0.2565, 0.2761)


def get_loaders(data_dir="./data", batch_size=128, num_workers=4):
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        # AutoAugment policy tuned for CIFAR
        AutoAugment(policy=AutoAugmentPolicy.CIFAR10),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
        # Cutout (erasing) as final step
        transforms.RandomErasing(p=0.5, scale=(0.02, 0.25), ratio=(0.3, 3.3)),
    ])

    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
    ])

    train_ds = datasets.CIFAR100(data_dir, train=True,  transform=train_tf, download=True)
    test_ds  = datasets.CIFAR100(data_dir, train=False, transform=test_tf,  download=True)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True, persistent_workers=True)
    test_loader  = DataLoader(test_ds,  batch_size=256,         shuffle=False,
                              num_workers=num_workers, pin_memory=True, persistent_workers=True)
    return train_loader, test_loader


# ─────────────────────────────────────────────────────────────────────────────
# 4.  Learning-rate schedule
# ─────────────────────────────────────────────────────────────────────────────

def cosine_lr_with_warmup(optimizer, epoch, warmup_epochs, total_epochs, base_lr, min_lr=1e-5):
    """Linear warmup → cosine decay.  Called once per epoch."""
    if epoch < warmup_epochs:
        lr = base_lr * (epoch + 1) / warmup_epochs
    else:
        progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
        lr = min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * progress))
    for pg in optimizer.param_groups:
        pg["lr"] = lr
    return lr


# ─────────────────────────────────────────────────────────────────────────────
# 5.  Training loop
# ─────────────────────────────────────────────────────────────────────────────

class AverageMeter:
    def __init__(self):
        self.reset()
    def reset(self):
        self.val = self.avg = self.sum = self.count = 0
    def update(self, val, n=1):
        self.val   = val
        self.sum  += val * n
        self.count += n
        self.avg   = self.sum / self.count


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total   += y.size(0)
    return 100.0 * correct / total


def train_one_epoch(model, loader, optimizer, criterion, device, args):
    model.train()
    loss_meter = AverageMeter()
    correct = total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        # Randomly apply MixUp or CutMix
        use_mix = random.random() < args.mix_prob
        if use_mix:
            if random.random() < 0.5:
                x, y_a, y_b, lam = mixup_data(x, y, alpha=args.mixup_alpha)
            else:
                x, y_a, y_b, lam = cutmix_data(x, y, alpha=args.cutmix_alpha)
            logits = model(x)
            loss   = mixed_criterion(criterion, logits, y_a, y_b, lam)
        else:
            logits = model(x)
            loss   = criterion(logits, y)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        # Gradient clipping helps early training stability
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        loss_meter.update(loss.item(), x.size(0))
        if not use_mix:
            correct += (logits.detach().argmax(1) == y).sum().item()
            total   += y.size(0)

    train_acc = 100.0 * correct / total if total > 0 else 0.0
    return loss_meter.avg, train_acc


def train(args):
    device = torch.device("cuda" if torch.cuda.is_available() else
                          "mps"  if torch.backends.mps.is_available() else "cpu")
    print(f"Device : {device}")

    # Seed for reproducibility
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    random.seed(args.seed)

    train_loader, test_loader = get_loaders(args.data_dir, args.batch_size, args.workers)

    model = ResNetCIFAR(arch=args.arch, num_classes=100,
                        dropout=args.block_dropout, head_dropout=args.head_dropout)
    model = model.to(device)

    # Count parameters
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Architecture : {args.arch}  ({n_params/1e6:.2f}M parameters)")

    # Label smoothing loss — crucial for CIFAR-100
    criterion = nn.CrossEntropyLoss(label_smoothing=args.label_smoothing)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=args.lr,
        momentum=0.9,
        weight_decay=args.weight_decay,
        nesterov=True,
    )

    best_acc = 0.0
    ckpt_path = Path(args.save_dir) / f"{args.arch}_best.pth"
    ckpt_path.parent.mkdir(parents=True, exist_ok=True)

    print(f"\n{'Epoch':>6} {'LR':>8} {'Train loss':>11} {'Train acc':>10} {'Test acc':>9} {'Best':>7}  {'Time':>7}")
    print("-" * 72)

    for epoch in range(args.epochs):
        t0 = time.time()
        lr = cosine_lr_with_warmup(optimizer, epoch, args.warmup_epochs,
                                   args.epochs, args.lr)
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer,
                                                criterion, device, args)
        test_acc = evaluate(model, test_loader, device)

        if test_acc > best_acc:
            best_acc = test_acc
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_acc": best_acc,
                "args": vars(args),
            }, ckpt_path)
            marker = " ✓"
        else:
            marker = ""

        elapsed = time.time() - t0
        print(f"{epoch+1:>6} {lr:>8.5f} {train_loss:>11.4f} {train_acc:>9.2f}% "
              f"{test_acc:>8.2f}% {best_acc:>6.2f}%  {elapsed:>5.1f}s{marker}")

    print(f"\nTraining complete.  Best test accuracy: {best_acc:.2f}%")
    print(f"Checkpoint saved to: {ckpt_path}")
    return best_acc


# ─────────────────────────────────────────────────────────────────────────────
# 6.  CLI
# ─────────────────────────────────────────────────────────────────────────────

def parse_args():
    p = argparse.ArgumentParser(description="ResNet on CIFAR-100")
    p.add_argument("--arch",           default="resnet34_cifar",
                   choices=list(ResNetCIFAR.CONFIGS.keys()),
                   help="Model variant")
    p.add_argument("--epochs",         type=int,   default=200)
    p.add_argument("--batch-size",     type=int,   default=128,  dest="batch_size")
    p.add_argument("--lr",             type=float, default=0.1)
    p.add_argument("--weight-decay",   type=float, default=5e-4, dest="weight_decay")
    p.add_argument("--warmup-epochs",  type=int,   default=5,    dest="warmup_epochs")
    p.add_argument("--label-smoothing",type=float, default=0.1,  dest="label_smoothing")
    p.add_argument("--block-dropout",  type=float, default=0.0,  dest="block_dropout",
                   help="Dropout2d between residual conv layers (0 = off)")
    p.add_argument("--head-dropout",   type=float, default=0.3,  dest="head_dropout",
                   help="Dropout before final FC layer")
    p.add_argument("--mix-prob",       type=float, default=0.5,  dest="mix_prob",
                   help="Probability of applying MixUp or CutMix")
    p.add_argument("--mixup-alpha",    type=float, default=0.4,  dest="mixup_alpha")
    p.add_argument("--cutmix-alpha",   type=float, default=1.0,  dest="cutmix_alpha")
    p.add_argument("--data-dir",       default="./data",          dest="data_dir")
    p.add_argument("--save-dir",       default="./checkpoints",   dest="save_dir")
    p.add_argument("--workers",        type=int,   default=4)
    p.add_argument("--seed",           type=int,   default=42)
    return p.parse_args()


import sys
from argparse import Namespace

args = Namespace(
    arch           = "wrn28_cifar",   # change to resnet34_cifar if preferred
    epochs         = 200,
    batch_size     = 128,
    lr             = 0.1,
    weight_decay   = 5e-4,
    warmup_epochs  = 5,
    label_smoothing= 0.1,
    block_dropout  = 0.0,
    head_dropout   = 0.3,
    mix_prob       = 0.5,
    mixup_alpha    = 0.4,
    cutmix_alpha   = 1.0,
    data_dir       = "./data",
    save_dir       = "./checkpoints",
    workers        = 2,   # Kaggle works best with 2
    seed           = 42,
)

train(args)